# **Imports**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, BatchNormalization
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pandas as pd
import os
import pickle
import matplotlib.ticker as ticker
import seaborn as sns
import tensorflow_probability as tfp
import warnings 
import random

# **Random Seed**

In [ ]:
def set_random_seeds(seed=42):
    
    # Python random
    random.seed(seed)
    
    # NumPy random
    np.random.seed(seed)
    
    # TensorFlow random
    tf.random.set_seed(seed)
    
    print(f"All random seeds set to: {seed}")

RANDOM_SEED = 21
set_random_seeds(RANDOM_SEED)

# **Constants**

In [ ]:
# Material constants    
fy = 60000 # Steel yield strength (psi)
gamma_c = 150 # Unit weight of normal weight concrete (lb/ft^3)
gamma_s = 490 # Unit weight of steel (lb/ft^3)
phi = 0.9

# Rebar Size
rebar_sizes = { # Rebar size number : Diameter (inches)
    6: 0.750,
    7: 0.875,
    8: 1.000,
    9: 1.128,
    10: 1.270
}

rebar_areas = {k: np.pi * (v / 2) ** 2 for k, v in rebar_sizes.items()} 


# PPO Hyperparameters
BUFFER_CAPACITY = 2048
BATCH_SIZE = 256
GAMMA = 0.99 # Discount factor for future rewards
GAE_LAMBDA = 0.95 # Lambda parameter for Generalized Advantage Estimation
CLIP_RATIO = 0.2 # Clipping parameter for PPO objective function
POLICY_LR = 3e-4 # Learning rate for the policy network (Actor)
VALUE_LR = 1e-3 # Learning rate for the value network (Critic)
ENTROPY_COEF = 0.03 # Entropy coefficient to encourage exploration
EPOCHS = 10 # Number of PPO optimization epochs per training iteration
STEP_SIZE = 10 #Number of design in each episode

# **Engineering Calculations (ACI 318-19)**

In [ ]:
def calculate_wu(DL, LL):
    return ((1.2 * DL) + (1.6 * LL)) # Calculate factored load (kip/ft)

def calculate_Mu_wo_sw(wu, L):
    return ((wu * (L ** 2)) / 8)

def calculate_ws_sw(b, d, As):
    return (((b * d - As) * gamma_c) + (As * gamma_s))/144000

def calculate_Mu(Mu_wo_sw, ws_sw, L):
    return (Mu_wo_sw + (1.2 * ws_sw * (L ** 2) / 8))

def calculate_a(As, fy, b, fc):
    return As * fy / (0.85 * fc * b)

def calculate_phiMn(As, fy, d, a):
    return (phi * As * fy * (d - a / 2))/12000

def calculate_et(d, a, beta1): #Calculate tensile strain in steel
    return 0.003 * ((d - (a/beta1)) * (beta1 / a))

def calculate_preq(Mu, phiMn, b, d, fc):
    Rn = Mu / (phi * b * (d ** 2))
    return 0.85 * (fc / fy) * (1 - np.sqrt(1 - ((2 * Rn) / (0.85 * fc))))

def calculate_pmin(fy, fc):
    # Handle negative or NaN fc values
    if np.isnan(fc) or fc <= 0:
        fc = 3000  # Default concrete strength in psi
        print(f"Warning: Invalid fc value encountered, using default fc = {fc} psi")
    
    sqrt_fc = np.sqrt(fc)
    
    # Handle potential NaN in sqrt calculation
    if np.isnan(sqrt_fc):
        sqrt_fc = np.sqrt(3000)  # Default sqrt(3000)
        print(f"Warning: NaN encountered in sqrt(fc), using default sqrt(3000)")
    
    return max(((3 * sqrt_fc) / fy), 200 / fy)

# **Visualization**

In [ ]:
# Visualization setup
plt.style.use('seaborn-white')
sns.set_context("paper", font_scale=1.2)

def update_plot(all_action_plot, rewards_history, concrete_mix_history, episode, co2_emissions_history, episode_mean_rewards):
    
    clear_output(wait=True)
    num_concrete_plots = 8
    n_rows, n_cols = 5, 3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 16), dpi=300)
    axes = axes.flatten()

    titles = [
        'Step-wise Reward', 'Episode Mean Reward', 'Depth', 'Width', 'Rebar Size', 'Number of Rebar',
        'Cement', 'Water', 'Fly Ash', 'Slag', 'Silica Fume', 'Fine Aggregate',
        'Coarse Aggregate', 'Superplasticizer', 'CO2 Emissions'
    ]
    
    palette = sns.color_palette("bright", len(titles))
    
    for i, ax in enumerate(axes):
        ax.clear()
        
        if i == 0:
            # Step-wise reward scatter plot
            ax.scatter(range(len(rewards_history)), rewards_history,
                       color=palette[i], s=3, alpha=0.8, edgecolor='k', linewidth=0.3)
            ax.set_ylabel('Step-wise Reward', fontsize=12)
            ax.set_ylim(-4, None)

        elif i == 1:
            # Episode mean reward line plot
            if episode_mean_rewards:
                ax.plot(range(len(episode_mean_rewards)), episode_mean_rewards,
                       color=palette[i], linewidth=1.5, alpha=0.8)
                ax.set_ylabel('Episode Mean Reward', fontsize=12)
                ax.set_ylim(-4, None)

        elif 2 <= i < 6:
            # Action parameter scatter plots
            action_data = all_action_plot[i - 2]
            ax.scatter(range(len(action_data)), action_data,
                       color=palette[i], s=3, alpha=0.8, edgecolor='k', linewidth=0.3)
            if i == 2:
                ax.set_ylabel('Depth (in)', fontsize=12)
            if i == 3:
                ax.set_ylabel('Width (in)', fontsize=12)
            if i == 4:
                ax.set_ylabel('Main Rebar Size', fontsize=12)
            if i == 5:
                ax.set_ylabel('Number of Main Rebar', fontsize=12)
                
        elif 6 <= i < 6 + num_concrete_plots:
            # Concrete mix scatter plots
            idx = i - 6
            mix_data = [mix[idx] for mix in concrete_mix_history]
            ax.scatter(range(len(mix_data)), mix_data,
                       color=palette[i], s=3, alpha=0.8, edgecolor='k', linewidth=0.3)
            if i == 6:
                ax.set_ylabel('Cement (lb/ft³)', fontsize=12)
            if i == 7:
                ax.set_ylabel('Water (lb/ft³)', fontsize=12)
            if i == 8:
                ax.set_ylabel('Fly Ash (lb/ft³)', fontsize=12)
            if i == 9:
                ax.set_ylabel('Slag (lb/ft³)', fontsize=12)
            if i == 10:
                ax.set_ylabel('Silica Fume (lb/ft³)', fontsize=12)
            if i == 11:
                ax.set_ylabel('Fine Aggregate (lb/ft³)', fontsize=12)
            if i == 12:
                ax.set_ylabel('Coarse Aggregate (lb/ft³)', fontsize=12)
            if i == 13:
                ax.set_ylabel('Superplasticizer (lb/ft³)', fontsize=12)
                
        elif i == 14:
            # CO2 emissions scatter plot
            ax.scatter(range(len(co2_emissions_history)), co2_emissions_history,
                       color=palette[i], s=3, alpha=0.8, edgecolor='k', linewidth=0.3)
            ax.set_ylabel('CO2 Emissions (lb/ft³)', fontsize=12)
        else:
            ax.axis('off')
            continue

        ax.set_xlabel('Step' if i != 1 else 'Episode', fontsize=12)
        ax.tick_params(axis='both', which='both', labelsize=10)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(5))
        ax.yaxis.set_major_locator(ticker.MaxNLocator(5))
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
        ax.minorticks_on()

    plt.tight_layout()
    plt.draw()
    plt.pause(0.001)
    
    fig.savefig(f'plot_episode_{episode:04d}.png', format='png')

# **PPO Buffer**

In [ ]:
class PPOBuffer:
    def __init__(self, state_dim, action_dim, buffer_capacity=BUFFER_CAPACITY, gamma=GAMMA, lam=GAE_LAMBDA):
        self.state_buffer = np.zeros((buffer_capacity, state_dim), dtype=np.float32)
        self.action_buffer = np.zeros((buffer_capacity, action_dim), dtype=np.float32)
        self.reward_buffer = np.zeros(buffer_capacity, dtype=np.float32)
        self.value_buffer = np.zeros(buffer_capacity, dtype=np.float32)
        self.advantage_buffer = np.zeros(buffer_capacity, dtype=np.float32)
        self.return_buffer = np.zeros(buffer_capacity, dtype=np.float32)
        self.logprob_buffer = np.zeros(buffer_capacity, dtype=np.float32)
        self.gamma, self.lam = gamma, lam
        self.buffer_capacity = buffer_capacity
        self.pointer = 0
        self.trajectory_start_idx = 0

    def store(self, state, action, reward, value, logprob):
        self.state_buffer[self.pointer] = state
        self.action_buffer[self.pointer] = action
        self.reward_buffer[self.pointer] = reward
        self.value_buffer[self.pointer] = value
        self.logprob_buffer[self.pointer] = logprob
        self.pointer += 1

    def finish_trajectory(self, last_value=0):
        path_slice = slice(self.trajectory_start_idx, self.pointer)
        rewards = np.append(self.reward_buffer[path_slice], last_value)
        values = np.append(self.value_buffer[path_slice], last_value)
        
        # GAE-Lambda advantage calculation
        deltas = rewards[:-1] + self.gamma * values[1:] - values[:-1]
        self.advantage_buffer[path_slice] = self._discounted_cumsum(deltas, self.gamma * self.lam)
        
        self.return_buffer[path_slice] = self._discounted_cumsum(rewards, self.gamma)[:-1]
        self.trajectory_start_idx = self.pointer

    def _discounted_cumsum(self, x, discount):

        y = np.zeros_like(x)
        y[-1] = x[-1]
        for t in reversed(range(len(x) - 1)):
            y[t] = x[t] + discount * y[t + 1]
        return y
    
    def get(self):

        valid_indices = slice(0, self.pointer)
        advantage_mean = np.mean(self.advantage_buffer[valid_indices])
        advantage_std = np.std(self.advantage_buffer[valid_indices])
        advantage_normalized = (self.advantage_buffer[valid_indices] - advantage_mean) / (advantage_std + 1e-8)
        
        data = dict(
            states=self.state_buffer[valid_indices],
            actions=self.action_buffer[valid_indices],
            returns=self.return_buffer[valid_indices],
            logprobs=self.logprob_buffer[valid_indices],
            advantages=advantage_normalized
        )
        return {k: tf.convert_to_tensor(v, dtype=tf.float32) for k, v in data.items()}

    def clear(self):
        self.pointer, self.trajectory_start_idx = 0, 0

 # **PPO Actor and Critic Network**

In [ ]:
# PPO Policy Network
class ActorNetwork(tf.keras.Model):
    def __init__(self, action_dim):
        super(ActorNetwork, self).__init__()
        self.action_dim = action_dim
        self.dense1 = Dense(256, activation='relu')
        self.bn1 = BatchNormalization()
        self.dense2 = Dense(128, activation='relu')
        self.bn2 = BatchNormalization()
        
        # Mean and log_std outputs
        self.mean = Dense(action_dim, activation='tanh')
        self.log_std = tf.Variable(
            initial_value=-0.5 * np.ones(action_dim, dtype=np.float32),
            trainable=True, dtype=tf.float32
        )

    def call(self, state, training=True):
        x = self.dense1(state)
        x = self.bn1(x, training=training)
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        mean = self.mean(x)
        std = tf.exp(self.log_std)
        return mean, std
    
    def get_action(self, state, deterministic=False):
        state = tf.convert_to_tensor(state, dtype=tf.float32)
        if len(state.shape) == 1:
            state = tf.expand_dims(state, 0)

        mean, std = self.call(state)
        
        if deterministic:
            return mean[0].numpy()
        
        dist = tfp.distributions.Normal(mean, std)
        action = dist.sample()
        action = tf.clip_by_value(action, -1.0, 1.0)
        
        logprob = dist.log_prob(action)
        logprob = tf.reduce_sum(logprob, axis=-1)
        
        return action[0].numpy(), logprob[0].numpy()
    
    def log_prob(self, state, action):
        mean, std = self.call(state)
        dist = tfp.distributions.Normal(mean, std)
        logprob = dist.log_prob(action)
        return tf.reduce_sum(logprob, axis=-1)

    def entropy(self, state):
        _, std = self.call(state)
        dist = tfp.distributions.Normal(tf.zeros_like(std), std)
        return tf.reduce_sum(dist.entropy(), axis=-1)

# PPO Value Network
class CriticNetwork(tf.keras.Model):
    def __init__(self):
        super(CriticNetwork, self).__init__()
        self.dense1 = Dense(256, activation='relu')
        self.bn1 = BatchNormalization()
        self.dense2 = Dense(128, activation='relu')
        self.bn2 = BatchNormalization()
        self.output_layer = Dense(1, dtype=tf.float32)

    def call(self, state, training=True):
        x = self.dense1(state)
        x = self.bn1(x, training=training)
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        value = self.output_layer(x)
        return tf.squeeze(value, axis=-1)

# **Agent**

In [ ]:
# PPO Agent
class PPOAgent:
    def __init__(self, state_dim, action_dim, total_episodes):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.total_episodes = total_episodes
        
        # Initialize networks
        self.actor = ActorNetwork(action_dim)
        self.critic = CriticNetwork()
        
        # Optimizers
        self.actor_optimizer = tf.keras.optimizers.Adam(learning_rate=POLICY_LR)
        self.critic_optimizer = tf.keras.optimizers.Adam(learning_rate=VALUE_LR)
        
        # Initialize buffer
        self.buffer = PPOBuffer(state_dim, action_dim)
        
        # Training tracking
        self.current_episode = 0
        sample_state = tf.zeros((1, state_dim), dtype=tf.float32)
        _ = self.actor(sample_state)
        _ = self.critic(sample_state)

    def get_action(self, state, deterministic=False):
        return self.actor.get_action(state, deterministic)

    def remember(self, state, action, reward, value, logprob):
        self.buffer.store(state, action, reward, value, logprob)

    def finish_trajectory(self, last_value=0):
        self.buffer.finish_trajectory(last_value)

    def learn(self):
        # Get data from buffer
        data = self.buffer.get()
        
        # Multiple epochs of updating
        for _ in range(EPOCHS):
            indices = np.random.permutation(len(data['states']))
            
            # Mini-batch training
            for start_idx in range(0, len(indices), BATCH_SIZE):
                end_idx = min(start_idx + BATCH_SIZE, len(indices))
                mini_batch_indices = indices[start_idx:end_idx]
                
                tf_indices = tf.convert_to_tensor(mini_batch_indices, dtype=tf.int32)
                
                mini_batch = {k: tf.gather(v, tf_indices) for k, v in data.items()}
                
                # Update policy and value networks
                self._update_networks(mini_batch)
        
        self.buffer.clear()

    def _update_networks(self, mini_batch):
        states = mini_batch['states']
        actions = mini_batch['actions']
        returns = mini_batch['returns']
        old_logprobs = mini_batch['logprobs']
        advantages = mini_batch['advantages']
        
        with tf.GradientTape() as actor_tape, tf.GradientTape() as critic_tape:
            # Critic loss
            values = self.critic(states)
            critic_loss = tf.reduce_mean(tf.square(returns - values))
            
            # Actor loss
            new_logprobs = self.actor.log_prob(states, actions)
            ratio = tf.exp(new_logprobs - old_logprobs)
            
            # Clipped surrogate objective
            clip_adv = tf.clip_by_value(ratio, 1.0 - CLIP_RATIO, 1.0 + CLIP_RATIO) * advantages
            actor_loss = -tf.reduce_mean(tf.minimum(ratio * advantages, clip_adv))
            
            # Add entropy bonus
            entropy = tf.reduce_mean(self.actor.entropy(states))
            actor_loss -= ENTROPY_COEF * entropy
        
        actor_gradients = actor_tape.gradient(actor_loss, self.actor.trainable_variables)
        critic_gradients = critic_tape.gradient(critic_loss, self.critic.trainable_variables)
        
        self.actor_optimizer.apply_gradients(zip(actor_gradients, self.actor.trainable_variables))
        self.critic_optimizer.apply_gradients(zip(critic_gradients, self.critic.trainable_variables))

# **Environment**

In [ ]:
# RC Beam Environment
class RC_Beam_Env:
    def __init__(self):
        # Initialize state parameters
        self.state = np.zeros(26)                                          
        self.trial_count = 0 
        self.Nm = 0
        
        self.loaded_model = tf.keras.models.load_model('/kaggle/input/model-save/1_concrete_prediction_model.keras')
        with open('/kaggle/input/model-save/3_x_scaler.pkl', 'rb') as f:
            self.x_scaler = pickle.load(f)
        with open('/kaggle/input/model-save/2_y_scaler_comp_strength.pkl', 'rb') as f:
            self.y_scaler_comp_strength = pickle.load(f)
        with open('/kaggle/input/model-save/4_y_scaler_co2.pkl', 'rb') as f:
            self.y_scaler_co2 = pickle.load(f)
           
    def reset(self, L, DL, LL):   #Resets the environment for a new design case with random initial state values
        
        self.L = L
        self.DL = DL
        self.LL = LL
        self.wu = calculate_wu(DL, LL)
        self.Mu_wo_sw = calculate_Mu_wo_sw(self.wu, L)
        self.state = np.zeros(26)
        
        self.state[0] = DL          # Dead load
        self.state[1] = LL          # Live load
        self.state[2] = L           # Beam length
        self.state[25] = self.Mu_wo_sw  # Moment without self weight
        
        # Random initialization for design parameters within ranges from convert_action
        self.state[3] = np.random.uniform(10, 20)  # b (width)
        self.state[4] = np.random.uniform(10, 50)  # d (depth)
        self.state[5] = np.random.choice([6, 7, 8, 9, 10])  # rebar_size
        self.state[6] = np.random.randint(2, 7)  # Nm (number of rebars)
        
        # Random initialization for concrete mix components within ranges from convert_action
        self.state[15] = np.random.uniform(3.5, 33)  # cement
        self.state[16] = np.random.uniform(4, 14)    # water
        self.state[17] = np.random.uniform(2, 14)    # fly_ash
        self.state[18] = np.random.uniform(5, 19)    # slag
        self.state[19] = np.random.uniform(0.8, 3.8) # silica_fume
        self.state[20] = np.random.uniform(20, 72)   # fine_aggregate
        self.state[21] = np.random.uniform(15, 81)   # coarse_aggregate
        self.state[22] = np.random.uniform(0.03, 0.65) # superplasticizer
        self.state[24] = np.random.choice([7, 28, 60, 90])  # Comp_Str_test_age
        
        
        # Calculate concrete properties f'c and CO2 emission
        fc, co2_emissions = self.Estimate_Concrete_Strength_and_CO2_Emission(
            self.state[15], self.state[16], self.state[17], self.state[18], 
            self.state[19], self.state[20], self.state[21], self.state[22], 
            self.state[24]
        )
        
        self.state[7] = fc  # concrete strength
        self.state[8] = co2_emissions  # CO2 emissions
        
        # Calculate additional parameters
        beta1 = 0.85 if fc <= 4000 else (0.85 - 0.05 * (fc - 4000) / 1000 if fc <= 8000 else 0.65)
        rebar_size = int(self.state[5])
        Nm = int(self.state[6])
        b = self.state[3]
        d = self.state[4]
        As = Nm * rebar_areas[rebar_size]
        a = calculate_a(As, fy, b, fc)
        ws_sw = calculate_ws_sw(b, d, As)
        phiMn = calculate_phiMn(As, fy, d, a)
        et = calculate_et(d, a, beta1)
        self.Mu = calculate_Mu(self.Mu_wo_sw, ws_sw, self.L)
        

        self.state[9] = a  # depth of equivalent rectangular stress block
        self.state[10] = As / (b * d)  # reinforcement ratio
        self.state[11] = phiMn
        self.state[12] = self.Mu
        self.state[13] = et  # tensile strain at extreme tension fiber
        self.state[14] = calculate_pmin(fy, fc)  # minimum reinforcement ratio
        
        self.trial_count = 0
        return self.state
        
    def step(self, action, L, DL, LL):
        
        # Convert normalized action values to design parameters                                                                        
        d, b, rebar_size, Nm, cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer, Comp_Str_test_age = self.convert_action(action)
        fc, co2_emissions = self.Estimate_Concrete_Strength_and_CO2_Emission(cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer, Comp_Str_test_age)

        if fc <= 4000:
            beta1 = 0.85
        elif fc <= 8000:
            beta1 = 0.85 - 0.05 * (fc - 4000) / 1000
        else:
            beta1 = 0.65
            
        # Calculate design parameters
        As = Nm * rebar_areas[rebar_size]
        dm = rebar_sizes[rebar_size]
        a = calculate_a(As, fy, b, fc)
        ws_sw = calculate_ws_sw(b, d, As)
        phiMn = calculate_phiMn(As, fy, d, a)
        et = calculate_et(d, a, beta1)
        self.Mu = calculate_Mu(self.Mu_wo_sw, ws_sw, self.L)
        preq = calculate_preq(self.Mu, phiMn, b, d, fc)
        pmin = calculate_pmin(fy, fc)
        self.Nm = Nm 

        # Update state parameters 
        self.state[0] = DL
        self.state[1] = LL
        self.state[2] = L
        self.state[3] = b
        self.state[4] = d
        self.state[5] = rebar_size
        self.state[6] = Nm
        self.state[7] = fc
        self.state[8] = co2_emissions
        self.state[9] = a
        self.state[10] = As / (b * d) 
        self.state[11] = phiMn
        self.state[12] = self.Mu
        self.state[13] = et
        self.state[14] = pmin
        self.state[15:23] = [cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer]
        self.state[24] = Comp_Str_test_age
        self.state[25] = self.Mu_wo_sw 
        
        # Calculate reward
        reward, R_neg_hard = self.calculate_reward(phiMn, self.Mu, preq, pmin, et, d, As, b, Nm, rebar_size, co2_emissions, fc,
                                     cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer)
    
        self.trial_count += 1

        done = self.trial_count == STEP_SIZE  # Episode is done after specified trials
        
        return self.state, np.float32(reward), done, R_neg_hard

    def convert_action(self, action):
        action = np.clip(action, -1, 1)
        
        # Handle NaN values by replacing with 0
        action = np.nan_to_num(action, nan=0.0)
        
        def smoother_map(x, min_val, max_val):
            return min_val + (max_val - min_val) * (x + 1) / 2
            
        d = smoother_map(action[0], 10, 50)
        
        b = smoother_map(action[1], 10, 20)
    
        rebar_sizes = [6, 7, 8, 9, 10]
    
        rebar_index_value = np.clip((action[2] + 1) * 2.5, 0, 4)
    
        if np.isnan(rebar_index_value):
            rebar_size = rebar_sizes[2]
        else:
            rebar_size_idx = int(rebar_index_value)
            rebar_size = rebar_sizes[rebar_size_idx]
    
        if np.isnan(smoother_map(action[3], 2, 6)):
            Nm = 4 
        else:
            Nm = int(smoother_map(action[3], 2, 6))
    
        water = smoother_map(action[5], 4, 14)
        cement = smoother_map(action[4], 3.5, 33)
        slag = smoother_map(action[7], 5, 19)
        fly_ash = smoother_map(action[6], 2, 14)
        silica_fume = smoother_map(action[8], 0.8, 3.8)
        fine_aggregate = smoother_map(action[9], 20, 72)
        coarse_aggregate = smoother_map(action[10], 15, 81)
        superplasticizer = smoother_map(action[11], 0.03, 0.65)
        
        Comp_Str_test_ages = [7, 28, 60, 90]
        
        # Handle NaN in test age calculation
        test_age_raw = (action[12] + 1) * 2
        if np.isnan(test_age_raw):
            test_age_idx = 1  # Default to 28 days
        else:
            test_age_idx = int(np.clip(test_age_raw, 0, 3))
        
        Comp_Str_test_age = Comp_Str_test_ages[test_age_idx] 
        
        return d, b, rebar_size, Nm, cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer, Comp_Str_test_age


    def Estimate_Concrete_Strength_and_CO2_Emission(self, cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer, Comp_Str_test_age):
        
        Fresh_Density = water + cement + slag + fly_ash + silica_fume + fine_aggregate + coarse_aggregate + superplasticizer
        
        new_sample = np.array([water, cement, slag, fly_ash, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer, Fresh_Density, Comp_Str_test_age]) 

        new_sample_reshaped = new_sample.reshape(1, -1) 

        new_sample_scaled = self.x_scaler.transform(new_sample_reshaped)

        prediction_scaled = self.loaded_model.predict(new_sample_scaled, verbose=0)

        predicted_comp_strength = self.y_scaler_comp_strength.inverse_transform(prediction_scaled[:, 0].reshape(-1, 1))
        
        predicted_co2 = self.y_scaler_co2.inverse_transform(prediction_scaled[:, 1].reshape(-1, 1))

        return predicted_comp_strength[0][0], predicted_co2[0][0]

    def calculate_reward(self, phiMn, Mu, preq, pmin, et, d, As, b, Nm, rebar_size, co2_emissions, fc,
                        cement, water, fly_ash, slag, silica_fume, fine_aggregate, coarse_aggregate, superplasticizer
                        ):
        # weights
        w = [0.8, 0.4, 0.6, 0.4, 0.2, 0.2, 1.0, 0.5]
    
        R_neg_hard = 0
        p = As / (b * d)
    
        # Hard constraints (Negative Reward )
        if p < preq:
            R_neg_hard -= w[0] * ((preq - p) / preq)
        if p < pmin:
            R_neg_hard -= w[1] * ((pmin - p) / pmin)
        if et < 0.005:
            R_neg_hard -= w[2] * ((0.005 - et) / 0.005)
        if d < 2.5 * b:
            R_neg_hard -= w[3] * ((2.5 * b - d) / (2.5 * b))
        if d < self.L / 16:
            R_neg_hard -= w[4] * ((self.L / 16 - (d + 2)) / (self.L / 16))
        if phiMn < Mu:
            R_neg_hard -= w[6] * ((Mu - phiMn) / Mu)

        total_mass = cement + water + fly_ash + slag + silica_fume + fine_aggregate + coarse_aggregate + superplasticizer
        if not (110 <= total_mass <= 170):
            R_neg_hard -= 0.2
    
        # Water to Binder ratio penalty
        # Calculate binder mass (cement + fly_ash + slag + silica_fume)
        binder = self.state[15] + self.state[17] + self.state[18] + self.state[19]
        # Calculate w/b ratio
        w_b_ratio = self.state[16] / binder
        
        # Define max and min w/b ratio
        w_b_min = 0.3
        w_b_max = 0.6
        
        # Calculate penalty for w/b ratio outside the desired range
        if w_b_ratio < w_b_min:
            penalty = (w_b_min - w_b_ratio) / w_b_min
            R_neg_hard -= w[7] * penalty
        elif w_b_ratio > w_b_max:
            penalty = (w_b_ratio - w_b_max) / w_b_max
            R_neg_hard -= w[7] * penalty
    
        R_pos = 0
        if R_neg_hard == 0:
            R_pos = 2 * np.exp(-0.06 * co2_emissions) #Positive Reward 
    
        reward = R_neg_hard  + R_pos
    
        return reward , R_neg_hard

# **Train Loop**

In [ ]:
# Training Loop with Excel Data Export
warnings.simplefilter(action='ignore', category=UserWarning)

successful_designs = []
d_action = []
b_action = []
rebar_size_action = []
Nm_action = []
all_action_plot = [d_action, b_action, rebar_size_action, Nm_action]

env = RC_Beam_Env() # Create RC beam environment instance
state_dim = 26  # State space dimension
action_dim = 13 # Action space dimension
total_episodes = 3001

agent = PPOAgent(state_dim, action_dim, total_episodes=total_episodes)

rewards_history = [] 
concrete_mix_history = [] 
co2_emissions_history = []
episode_mean_rewards = []  # List to store mean rewards per episode


cement_history = []
water_history = []
fly_ash_history = []
slag_history = []
silica_fume_history = []
fine_aggregate_history = []
coarse_aggregate_history = []
superplasticizer_history = []

model_dir = "/kaggle/working/saved_models"
os.makedirs(model_dir, exist_ok=True)

def save_all_plot_data_to_excel(episode_num, seed=RANDOM_SEED):
    
    plot_data = {
        'Seed': [seed] * len(rewards_history),
        'Step': range(len(rewards_history)),
        'Step_Reward': rewards_history,
        'Depth_in': d_action,
        'Width_in': b_action,
        'Rebar_Size': rebar_size_action,
        'Number_of_Rebars': Nm_action,
        'CO2_Emissions_lb_per_ft3': co2_emissions_history,
        'Cement_lb_per_ft3': cement_history,
        'Water_lb_per_ft3': water_history,
        'Fly_Ash_lb_per_ft3': fly_ash_history,
        'Slag_lb_per_ft3': slag_history,
        'Silica_Fume_lb_per_ft3': silica_fume_history,
        'Fine_Aggregate_lb_per_ft3': fine_aggregate_history,
        'Coarse_Aggregate_lb_per_ft3': coarse_aggregate_history,
        'Superplasticizer_lb_per_ft3': superplasticizer_history
    }
    
    # Create DataFrame for step-wise data
    df_plot_data = pd.DataFrame(plot_data)
    
    # Create DataFrame for episode mean rewards with seed
    df_episode_rewards = pd.DataFrame({
        'Seed': [seed] * len(episode_mean_rewards),
        'Episode': range(len(episode_mean_rewards)),
        'Episode_Mean_Reward': episode_mean_rewards
    })
    
    # Save to Excel with seed in filename
    filename = f'all_plot_data_seed_{seed}_episode_{episode_num}.xlsx'
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_plot_data.to_excel(writer, sheet_name='Step_Wise_Data', index=False)
        df_episode_rewards.to_excel(writer, sheet_name='Episode_Mean_Rewards', index=False)
        
        
        if successful_designs:
            df_successful = pd.DataFrame(successful_designs)
            df_successful['Seed'] = seed  
            df_successful.to_excel(writer, sheet_name='Successful_Designs', index=False)
    
    print(f"All plot data saved to '{filename}'")
    print(f"Random seed: {seed}")
    print(f"Total steps recorded: {len(rewards_history)}")
    print(f"Total episodes recorded: {len(episode_mean_rewards)}")
    print(f"Total successful designs: {len(successful_designs)}")

for episode in range(total_episodes):
    L = np.random.uniform(15, 30)
    DL = np.random.uniform(1.5, 4.5)
    LL = np.random.uniform(1.5, 4.5)
    state = env.reset(L, DL, LL)
    agent.current_episode = episode
    
    episode_rewards = [] 
    
    for t in range(STEP_SIZE):
        # Get action from PPO agent
        action, logprob = agent.get_action(state)

        value = agent.critic(tf.convert_to_tensor([state], dtype=tf.float32)).numpy()[0]
        
        # Take action in environment
        next_state, reward, done, R_neg_hard = env.step(action, L, DL, LL)
        
        # Store in PPO buffer
        agent.remember(state, action, reward, value, logprob)
        
        # Record data for plotting
        d_action.append(env.state[4])  # depth
        b_action.append(env.state[3])  # width
        rebar_size_action.append(env.state[5])  # rebar size
        Nm_action.append(env.state[6])  # number of rebars
        concrete_mix_history.append(env.state[15:23].tolist())  # concrete mix components
        co2_emissions_history.append(env.state[8])
        rewards_history.append(reward)
        episode_rewards.append(reward)  # Track reward for this step in episode
        
        # Record individual concrete mix components for Excel export
        cement_history.append(env.state[15])
        water_history.append(env.state[16])
        fly_ash_history.append(env.state[17])
        slag_history.append(env.state[18])
        silica_fume_history.append(env.state[19])
        fine_aggregate_history.append(env.state[20])
        coarse_aggregate_history.append(env.state[21])
        superplasticizer_history.append(env.state[22])
        
        # Save successful designs
        if R_neg_hard == 0:
            successful_designs.append({
                'Episode': episode,
                'Step': len(rewards_history) - 1,
                'DL': DL,
                'LL': LL,
                'Beam Length (L)': L,
                'Width (b)': env.state[3],
                'Depth (d)': env.state[4],
                'Rebar Size': env.state[5],
                'Number of Rebars (Nm)': env.state[6],
                'fc': env.state[7],
                'co2_emissions': env.state[8],
                'a': env.state[9],
                'As / (b * d)': env.state[10],
                'PhiMn': env.state[11],
                'Mu': env.state[12],
                'et': env.state[13],
                'pmin': env.state[14],
                'cement': env.state[15],
                'water': env.state[16],
                'fly_ash': env.state[17],
                'slag': env.state[18],
                'silica_fume': env.state[19],
                'fine_aggregate': env.state[20],
                'coarse_aggregate': env.state[21],
                'superplasticizer': env.state[22],
                'test_age': env.state[24],
                'Reward': reward
            })
        
        
        state = next_state
        
        if done:
            final_value = agent.critic(tf.convert_to_tensor([next_state], dtype=tf.float32)).numpy()[0]
            agent.finish_trajectory(final_value)
            
            episode_mean_reward = np.mean(episode_rewards)
            episode_mean_rewards.append(episode_mean_reward)
            break
    
    agent.learn()
    
   
    if episode % 10 == 0:
        print(f"Episode: {episode}")
    
    # Update plots and save data periodically
    if episode % 50 == 0:
        update_plot(all_action_plot, rewards_history, concrete_mix_history, episode, co2_emissions_history, episode_mean_rewards)
        print(f"Episode: {episode}")
        
        save_all_plot_data_to_excel(episode)
        
        if successful_designs:
            df_successful = pd.DataFrame(successful_designs)
            df_successful.to_excel("successful_designs.xlsx", index=False)
            print("Number of successful designs:", df_successful.shape[0])
        
        if episode_mean_rewards:
            df_episode_rewards = pd.DataFrame({
                'Episode': range(len(episode_mean_rewards)),
                'Mean_Reward': episode_mean_rewards
            })
            df_episode_rewards.to_excel("episode_mean_rewards.xlsx", index=False)
            print(f"Episode mean rewards saved. Latest mean reward: {episode_mean_rewards[-1]:.4f}")

    # Save model weights periodically
    if episode % 100 == 0:
        actor_filename = os.path.join(model_dir, f"ppo_actor_model_episode_{episode}.weights.h5")
        critic_filename = os.path.join(model_dir, f"ppo_critic_model_episode_{episode}.weights.h5")
        agent.actor.save_weights(actor_filename)
        agent.critic.save_weights(critic_filename)

# **Test**

In [ ]:
# It returns the three best designs that are similar to the user's condition.
from scipy.spatial import KDTree
import numpy as np
import pandas as pd

L = float(input("Enter Beam Length (in ft): "))
DL = float(input("Enter Dead Load (in kip/ft): "))
LL = float(input("Enter Live Load (in kip/ft): "))

def find_top_n_designs(new_L, new_DL, new_LL, successful_designs, n=10):
    if not successful_designs:
        raise ValueError("No successful designs were saved during training.")
    
    # Extract L, DL, LL from successful designs
    L_values = [design['Beam Length (L)'] for design in successful_designs]
    DL_values = [design['DL'] for design in successful_designs]
    LL_values = [design['LL'] for design in successful_designs]
    
    # Find min and max for each parameter
    min_L, max_L = min(L_values), max(L_values)
    min_DL, max_DL = min(DL_values), max(DL_values)
    min_LL, max_LL = min(LL_values), max(LL_values)
    
    # Normalize the points
    normalized_points = []
    for design in successful_designs:
        L = design['Beam Length (L)']
        DL = design['DL']
        LL = design['LL']
        normalized_L = (L - min_L) / (max_L - min_L)
        normalized_DL = (DL - min_DL) / (max_DL - min_DL)
        normalized_LL = (LL - min_LL) / (max_LL - min_LL)
        normalized_points.append([normalized_L, normalized_DL, normalized_LL])
    
    # Build KD-tree with normalized points
    kdtree = KDTree(normalized_points)
    
    # Normalize the user input
    normalized_new_L = (new_L - min_L) / (max_L - min_L)
    normalized_new_DL = (new_DL - min_DL) / (max_DL - min_DL)
    normalized_new_LL = (new_LL - min_LL) / (max_LL - min_LL)
    new_point = [normalized_new_L, normalized_new_DL, normalized_new_LL]
    
    # Determine the number of designs to return
    num_designs = len(successful_designs)
    n = min(n, num_designs)
    
    # Query for the n nearest neighbors
    _, indices = kdtree.query(new_point, k=n)
    
    # Retrieve the designs and sort by reward
    top_n_designs = [successful_designs[i] for i in indices]
    top_n_designs.sort(key=lambda x: x['Reward'], reverse=True)
    
    # Calculate actual Euclidean distances for the sorted designs
    actual_points = np.array([[design['Beam Length (L)'], design['DL'], design['LL']] for design in top_n_designs])
    new_point_actual = np.array([new_L, new_DL, new_LL])
    actual_distances = np.sqrt(np.sum((actual_points - new_point_actual) ** 2, axis=1))
    
    return top_n_designs, actual_distances


successful_designs = pd.read_excel('/kaggle/working/successful_designs.xlsx').to_dict('records')

# Find the top 10 closest successful designs and their distances
top_10_designs, distances = find_top_n_designs(L, DL, LL, successful_designs, n=10)

# Only use the top 3 for display
top_3_designs = top_10_designs[:3]
top_3_distances = distances[:3]


designs_data = []
for i, (design, distance) in enumerate(zip(top_3_designs, top_3_distances)):
    design_dict = {
        "Input Parameters": {
            "Nearest Design": f"{i+1}",
            "Euclidean distance from Input": f"{distance:.4f}",
            "Design L (foot)": f"{design['Beam Length (L)']:.2f}",
            "Design DL (kip/ft)": f"{design['DL']:.2f}",
            "Design LL (kip/ft)": f"{design['LL']:.2f}"
        },
        "Beam Dimensions": {
            "Beam Depth (d) (inches)": f"{design['Depth (d)']:.2f}",
            "Beam Width (b) (inches)": f"{design['Width (b)']:.2f}",
            "Main Rebar Size (#)": f"{design['Rebar Size']:.0f}",
            "Number of Main Rebars": f"{design['Number of Rebars (Nm)']:.0f}"
        },
        "Material Properties and CO2 Emissions": {
            "Concrete Compressive Strength (f'c) (psi)": f"{design['fc']:.2f}",
            "CO2 Emissions (lb/ft³)": f"{design['co2_emissions']:.2f}"
        },
        "Design Parameters": {
            "Shear Span (a) (inches)": f"{design['a']:.2f}",
            "Reinforcement Ratio": f"{design['As / (b * d)']:.4f}",
            "Design Moment Capacity (φMn) (kip-ft)": f"{design['PhiMn']:.2f}",
            "Required Moment (Mu) (kip-ft)": f"{design['Mu']:.2f}",
            "Strain in Tension Steel (εt)": f"{design['et']:.4f}",
            "Minimum Reinforcement Ratio (ρmin)": f"{design['pmin']:.4f}",
            "Test Age (days)": f"{design['test_age']:.0f}"
        },
        "Mix Design": {
            "Cement (lb/ft³)": f"{design['cement']:.2f}",
            "Water (lb/ft³)": f"{design['water']:.2f}",
            "Fly Ash (lb/ft³)": f"{design['fly_ash']:.2f}",
            "Slag (lb/ft³)": f"{design['slag']:.2f}",
            "Silica Fume (lb/ft³)": f"{design['silica_fume']:.2f}",
            "Fine Aggregate (lb/ft³)": f"{design['fine_aggregate']:.2f}",
            "Coarse Aggregate (lb/ft³)": f"{design['coarse_aggregate']:.2f}",
            "Superplasticizer (lb/ft³)": f"{design['superplasticizer']:.3f}",
            "Water–Binder (w/b) ratios": f"{design['water'] / (design['cement'] + design['fly_ash'] + design['slag'] + design['silica_fume']):.4f}"
        },
        "Performance": {
            "Reward Achieved": f"{design['Reward']:.2f}",  
        }
    }
    designs_data.append(design_dict)


def display_designs(designs_data):
    # Define column widths
    param_width = 35
    value_width = 15
    total_width = param_width + 3 * (value_width + 3)
    
    # Print header
    print(f"{'='*total_width}")
    print(f"BEAM DESIGN RECOMMENDATIONS".center(total_width))
    print(f"{'='*total_width}")
    print(f"Input Parameters: L = {L} ft, DL = {DL} kip/ft, LL = {LL} kip/ft")
    print(f"{'-'*total_width}")
    
    # Print table header
    print(f"{'Parameter':<{param_width}} | {'Design #1':^{value_width}} | {'Design #2':^{value_width}} | {'Design #3':^{value_width}}")
    print(f"{'-'*param_width}-+-{'-'*value_width}-+-{'-'*value_width}-+-{'-'*value_width}")
    

    for section_name in designs_data[0].keys():
        print(f"\n{section_name.upper()}:")
        print(f"{'-'*total_width}")
        
        for param_name in designs_data[0][section_name].keys():
            row = f"{param_name:<{param_width}} | "
            for design in designs_data:
                row += f"{design[section_name][param_name]:^{value_width}} | "
            print(row)
        
        print(f"{'-'*total_width}")
    
display_designs(designs_data)